# Phase2: 拡張なしで学習してみる（ベースライン）

> ⏱ 所要時間: 説明を読む 10分 ＋ 実行 1〜2分（`QUICK = True` のとき）

## このNotebookのゴール
**データ拡張を使わずにAIを学習させ、「過学習」が実際に起きることを確認する。**
そして、拡張の効果を測るための**基準（ベースライン）**を作る。

### なぜ「拡張なし」を先にやるのか
拡張の効果は、**「やらなかった場合との差」**でしか測れません。
「拡張すると精度が80%になった」と言われても、拡張なしで70%なのか、79%なのかで、価値がまったく違います。
そこで、まず**拡張なし**の成績を測ります（これが**ベースライン** ＝ 比較の基準）。

## このNotebookでやること
1. AI（小さなCNN）の中身を見る
2. 拡張なしで学習し、**学習曲線**（学習の様子のグラフ）を描く
3. グラフから**過学習**を読み取る
4. 乱数の種（シード）を変えて、**結果がぶれる**ことを確認する

## キーワード（用語集は Notebook 00 にあります）
| 用語 | 意味 |
|---|---|
| **エポック** | 全ての学習用写真を1周すること |
| **loss（ロス）** | AIの間違いの大きさ。小さいほど良い |
| **過学習** | train には強いのに、val/test には弱い状態 |
| **gap（ギャップ）** | train 精度 − val 精度。大きいほど過学習 |

In [ ]:
import sys
from pathlib import Path

# 正しい環境（プロジェクトの .venv）で動いているか確認する
try:
    import torch  # noqa: F401
except ModuleNotFoundError:
    raise SystemExit(
        f"torch が見つかりません。今のPython: {sys.executable}\n"
        "→ カーネルを「Python (image_augment_lab)」（または プロジェクトの .venv/bin/python）に切り替えてください。"
    )

# src/ をimportできるようにする（notebooks/ からでもプロジェクト直下からでも動く）
sys.path.insert(0, str(next(p for p in [Path.cwd() / "src", Path.cwd().parent / "src"] if p.exists())))
from nb_utils import *

%matplotlib inline

## 設定: 実験の「規模」を選ぶ

- `QUICK = True`: **1回だけ**学習（短時間）。まずはこちらで流れを理解しましょう。
- `QUICK = False`: **3回**（シード0・1・2）学習して平均を取ります。**MANUAL.md の数値と同じ条件**ですが、時間は約3倍かかります。
- `EPOCHS = 60`: 全写真を**60周**学習します。`ep(60)` は、動作確認モード（`NB_FAST`）のとき自動で短くするための関数です。通常は気にしなくて構いません。

In [ ]:
QUICK = True                          # True: 1回（短時間）/ False: 3回（記載値と同条件）
SEEDS = [0] if QUICK else [0, 1, 2]   # 乱数の種のリスト。1つにつき1回学習する
EPOCHS = ep(60)                       # 学習の周回数

## 1. AI本体（SmallCNN）の中身

この演習では、**畳み込み3層**の**小さなCNN**を使います。次のセルで構造を表示します。

### CNNをざっくり理解する
画像を見るAIは、写真を**何段階かに分けて**処理します。

1. **畳み込み層（Conv）**: 小さな窓（3×3画素）を画像にずらしながら当て、「線・角・模様」などの**特徴**を見つける
2. **ReLU**: 不要な反応を消す
3. **MaxPool**: 画像を半分の大きさに縮め、重要な特徴だけ残す
4. これを3回繰り返し、最後の**全結合層**で「アリ or ハチ」を判定する

> 💡 **なぜわざと小さくするのか？** 小さいと数十秒で学習でき、**過学習の様子が見やすい**からです。
> 小さくても、アリ・ハチ程度の違いなら見分けられます。

次のセルの「パラメータ数」は、AIが学習で調整する**つまみの数**です。約2万4千個と、実用的なAI（数千万〜数億個）よりずっと小さいです。

In [ ]:
model = T.SmallCNN(2)       # 2 = 分類するクラス数（アリ・ハチ）
print(model)                # 構造を表示
print("パラメータ数:", sum(p.numel() for p in model.parameters()))

## 2. 拡張なしで学習する

`fit()` という関数が、**学習を1回**行います。引数の意味は次のとおりです。

| 引数 | 意味 |
|---|---|
| `aug="none"` | **拡張なし**（リサイズと正規化だけ） |
| `seed=0` | 乱数の種 |
| `epochs=EPOCHS` | 全写真を何周学習するか |

### 学習の中身（1エポックごと）
1. train の写真を使って学習する（間違いを減らす方向に、AIのつまみを少し調整する）
2. val の写真で「今の実力」を測る（**val は学習に使わない**）

これをエポック数だけ繰り返し、各エポックの **loss** と **精度** を記録します。
test の成績は、**学習が全部終わったあと、1回だけ**測ります（途中で test を見て調整するのは反則です）。

次のセルは学習して、**学習曲線**を描きます。

In [ ]:
r = fit(aug="none", seed=0, epochs=EPOCHS, verbose=False)   # 学習を1回実行
plot_history(r)      # 学習曲線を描く
summarize(r)         # 最終エポックの成績をまとめる

## 3. 学習曲線の読み方

グラフは2つあります。**横軸はエポック**（何周目か）です。

### 左: loss（間違いの大きさ。**小さいほど良い**）
- 青（train）: **下がり続けて**いれば、AIは学習用の写真をどんどん覚えている。
- オレンジ（val）: 学習に使っていない写真での間違い。

### 右: 精度（正解率。**大きいほど良い**）
- train 精度が上がっているのに、val 精度が**あまり上がらない**…これが過学習のサイン。

### 過学習のサイン（チェックリスト）
| 見るもの | 過学習のサイン |
|---|---|
| **loss のグラフ** | train は下がるのに、**val は下がらない、または上がる** |
| **精度のグラフ** | train が val を**大きく上回る**（この差が gap） |

> 🍎 たとえ話: train は「勉強した問題集の点数」、val は「初見の模試の点数」。問題集では満点に近づくのに模試が伸びない → 丸暗記している。

### val の線がギザギザなのはなぜ？
val の写真は**約76枚**と少ないので、**1枚の判定が変わるだけで精度が約1.3ポイント**動きます。そのため線がガタガタします。
**少ないデータでは、結果が不安定**です。（次の節で詳しく見ます）

## 4. 乱数の種を変えると、結果はどれくらいぶれる？

AIの学習では、**初期値**と**写真を見る順番**に乱数を使います。乱数の種（**シード**）を変えるだけで、結果が少し変わります。
とくにデータが少ないと、**ぶれが大きい**です。

だから、**条件を比べるときは、複数のシードで学習して平均を見る**のが本来のやり方です。
（1回の結果だけで「拡張Aが良い！」と言うのは、危険です）

次のセルは、`SEEDS` の数だけ学習して、結果を表にします。`QUICK = True` のときは1行だけです。

In [ ]:
df, results = run_seeds(SEEDS, aug="none", epochs=EPOCHS)
df       # 各シードの結果（train_acc/val_acc/test_acc/gap）

In [ ]:
mean_table(df)   # シード平均 ± ばらつき（標準偏差）

**表の見方**
- `train_acc / val_acc / test_acc`: それぞれの正解率（例: 0.80 ＝ 80%）。
- `gap`: train − val。**大きいほど過学習**。
- `±` の後ろの数字: **ばらつき**（標準偏差）。この数字が大きいと、「差があるように見えても誤差かも」と疑います。

## 参考: 記載値（MANUAL.md、60エポック・3シード平均）
- train **0.859** / val **0.719** / test **0.809** / gap 約 **0.14**
- 学習用の写真には86%正解、初めて見る写真（val）には72%。**約14ポイントの差＝過学習**。
- シード0だけ val が 0.671、シード1・2は 0.74〜0.75 と、**シードで約7ポイント違う**。これがぶれです。

## 🧪 試してみよう

### 演習1: 学習回数（エポック）を変える
**やること**: 設定のセルの **`EPOCHS = ep(60)`** を **`EPOCHS = ep(20)`** に書き換え、**設定のセル以降を実行し直し**ます。
次に **`ep(150)`** でもやってみてください。
- **見るポイント**: エポックが増えるほど、train 精度は上がる？ val 精度は？ **gap（差）は広がっていきませんか？**
- **分かること**: 長く学習するほど、train の写真を丸暗記して**過学習が進む**。（Notebook 03 で、これを拡張が防ぐ様子を見ます）

### 演習2: 3回の平均を見る
**やること**: 設定のセルの **`QUICK = True`** を **`False`** に書き換え、設定のセル以降を実行し直します（約3分）。
- **見るポイント**: 表の `±` の値を見てください。`val_acc` と `test_acc` のどちらがぶれていますか？
- **分かること**: 写真が少ないほど、**1回の結果は当てにならない**。

### 演習3: 学習の速さ（学習率）を変える
**やること**: 「2. 拡張なしで学習する」のセルの `fit(` の中に **`lr=0.01`** を足します。
`r = fit(aug="none", seed=0, epochs=EPOCHS, verbose=False, lr=0.01)` のようにして実行してください。
- **`lr`（学習率）**は、AIが**1回の更新でどれくらい大きく調整するか**。標準は `0.001`。
- **見るポイント**: 大きくすると（0.01）、グラフは**荒れやすい**？ 小さくすると（0.0001）、**学習が遅い**？

---

## ✅ まとめ

- **拡張なし**で60エポック学習すると、train 約0.86 / val 約0.72 / test 約0.81（記載値）。
- **train loss が下がるのに val loss が下がらない = 過学習**。ここを拡張で改善するのが以降のテーマ。
- 学習を長くするほど、過学習は進みやすい。
- 少ないデータでは**結果がぶれる**。条件の比較は**複数シードの平均**で行う。
- この結果が、以降の**比較の基準（ベースライン）**になる。

➡ 次は **`03_augment_vs_none.ipynb`**。**拡張あり・なしを同じ条件で比べ**、拡張が過学習を抑えるかを確かめます。